# part 2

* * In this method (bigram), if we consider a single character at a time, there are 27 possible contexts. If we consider two characters, there are 27*27 possibilities. There are too many rows in this matrix with too few counts for each possibility, causing the method to become unmanageable and ineffective. Therefore, we will implement a Multilayer Perceptron (MLP) to predict the next character in the sequence. We will follow the approach outlined in this paper: [Bengio et al., 2003](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf). While this paper uses a vocabulary of 17,000 words to build a word-level language model, we will use the same modeling approach for characters instead.
* * In this paper, a 17,000-word vocabulary is associated with a 30-dimensional feature vector for each word. This means each word is embedded in a 30-dimensional space, which is quite small for such a large vocabulary. Initially, these word embeddings are randomly distributed. However, as we tune these embeddings using backpropagation, some vectors will move closer together while others will move apart, reflecting their relationships and similarities. The paper uses a Multilayer Perceptron (MLP) to predict the next word based on previous words and trains the neural network by maximizing the log-likelihood.

In [ ]:
from IPython import display
display.Image(url = 'https://production-media.paperswithcode.com/Screen_Shot_2020-05-26_at_2.17.37_PM.png')


* * In this diagram, we are using the previous three words to predict the next word (the fourth word) in the sequence. The lookup table is a matrix with 17,000 words, each represented by a 30-dimensional vector, converting each index to its corresponding embedding vector. We have an input layer of 30 neurons for each of the three words, totaling 90 values.

* * The size of the hidden layer is a hyperparameter, meaning it's a design choice left to the designer. We will experiment with various hidden layer sizes to evaluate their performance. For example, with 100 neurons in the hidden layer, each neuron would be fully connected to the 90 input values, using a tanh function as the activation function. The output layer consists of 17,000 neurons, one for each word in the vocabulary, and all are fully connected to the hidden layer neurons.

* * There is a significant competition between the hidden layer and the output layer due to the number of parameters. Our parameters include the weights and biases of both the output and hidden layers, as well as the embedding lookup table. All these parameters will be tuned during backpropagation.

* * Let's implement this setup...

In [ ]:
import torch 
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# read in all the words
words = open('names.txt','r').read().splitlines()
words[:8]

In [ ]:
len(words)

In [ ]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

In [ ]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?
# to guess the letter after the 3rd letter

X, Y = [], []
for w in words[:5]:

    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

* finally dataset now looks like

In [ ]:
X.shape, X.dtype, Y.shape, Y.dtype

In [ ]:
X

In [ ]:
Y

* lets build the embedding lookup table C
* We have 27 characters, and we will embed them in a lower-dimensional space. In the paper, they have 17,000 words embedded in a 30-dimensional space. Since we only have 27 possible characters, let's start by embedding them in a space as small as 2 dimensions.

In [ ]:
C = torch.randn((27,2))
C

There are two different ways to handle the embedding process. The first method is to provide direct input to the C lookup table. The second method is to convert the input number to one-hot encoding and then input it to C. We will use the first method because it is faster.

For example, if our input is 5...

In [ ]:
# 1st way
C[5]

In [ ]:
# 2nd way
F.one_hot(torch.tensor(5), num_classes=27).float() @ C # num_classes is 27 because we have a 27 char

In [ ]:
C[[5,6,7]] # we can use like that pytorch idnexing is very flexible

In [ ]:
C[torch.tensor([5,6,7])]

In [ ]:
C[X].shape

In [ ]:
X[13,2] # index 13th second dimension value is 1

In [ ]:
C[1] == C[X[13,2]] # the equivalent of this value in the lookup table. you can verify manually in X

In [ ]:
emb = C[X]
emb.shape

In [ ]:
W1 = torch.randn((6,100)) # we have a 2 dimnesional embeddings and have a three of them so 2x3=6, number of layer depends our choice which is 100
b1 = torch.randn(100) # bias also randomly

In [ ]:
emb @ W1 + b1

* there was a error because we have to change shapes

In [ ]:
emb.shape 

* we need to convert 3d embedding to 2d, one way to do this

In [ ]:
torch.cat([emb[:,0,:], emb[:,1,:] , emb[:,2,:]],1) # 1 is cor dimension ???

* we have a problem. This code not generalize when we change the block_size this code will blow up
* there is a more efficient and dynamic way to do this(.view())

In [ ]:
torch.cat(torch.unbind(emb,1),1).shape # does not depend to block_size

* how can we fix tensor view?

In [ ]:
a = torch.arange(18)
print(f'a: {a}\n')
print(f'a.shape: {a.shape}\n',)
print(f'a.view(2,9): {a.view(2,9)} \n',)
print(f'a.view(3,2,3): {a.view(3,2,3)}\n',)

* The view() function is very powerful. It lets us reshape the tensor in any way, as long as the total number of elements stays the same.
* The view so efficient for us because every tensor represented in the computer a.storage and its always one dimensional vector but when we called .view we just manipulated some attributies of that tensor. one dimnesional sequence is interpreted to be n dimensional tensor. in this part no memory is being changed or copied or moved
* look here for more: http://blog.ezyang.com/2019/05/pytorch-internals/

In [ ]:
a.storage()

In [ ]:
print(emb.shape)
print(emb.view(32,6))

* 
In other words, in order to use the tensor in the size we want, we will use .view() and define the hidden layer again.

In [ ]:
h = emb.view(32,6) @ W1 +b1

In [ ]:
h.shape

In [ ]:
# Let's make this definition dynamic
h = emb.view(-1,6) @ W1 +b1

* PyTorch reads the -1 and infers the correct dimension based on the other dimensions provided. For example, if (2x3) is already used, and you have 32 times 6 (or inputs times 6), PyTorch will determine the appropriate shape.

In [ ]:
h = torch.tanh(emb.view(-1,6) @ W1 +b1)

* Due to the tanh function, our numbers are constrained to values between -1 and 1. The shape is 32 by 100,  which represents the hidden layer activations for each of our 32 examples.


In [ ]:
print(h.shape)
print(b1.shape)

* There's one more thing we need to be very careful with: broadcasting. In particular, we need to ensure that broadcasting behaves as expected. The shape of this tensor is 32 by 100, and the shape of the ones tensor is 100.
* We see that the addition will broadcast these two tensors, with the 32 by 100 tensor broadcasting to match the 100-dimensional tensor.
* 32,100 ---> 1,100 Broadcasting will align on the right, creating a fake dimension. This will turn the 100-dimensional tensor into a 1 by 100 row vector, which will then be vertically copied for each of the 32 rows, allowing for element-wise addition.
* In this case, the correct operation will occur because the same bias vector is added to all the rows of the matrix, which is what we want. It's always good practice to ensure this behavior.

In [ ]:
W2 = torch.randn((100, 27)) # 100 inputs, 27 output neurons
b2 = torch.randn(27)

* Finally, let's create the final layer. We'll define w2 and b2, where the input size is 100 and the output size will be 27, corresponding to the 27 possible characters.

In [ ]:
logits = h @ W2 +b2

In [ ]:
# Softmax transforms outputs into probabilities
counts = logits.exp()
prob = counts / counts.sum(1, keepdim=True)
print(prob.shape)
print(prob[0].sum())

In [ ]:
Y

* Now, similar to the previous video, we want to index into the rows of the probability matrix and, in each row, extract the probability assigned to the correct character as indicated.

In [ ]:
prob[torch.arange(32), Y]

* This provides the current probabilities assigned by the neural network, based on its weight settings, to the correct character in the sequence.
* You can see that the probabilities look reasonable for some characters (like this one, which is around 0.2), but they don't look very good for many other characters.
* The network currently considers some characters extremely unlikely. However, since we haven't trained the neural network yet, this will improve. Ideally, all these probabilities should be 1, indicating that we are correctly predicting the next character.

In [ ]:
# We want the average log probability across all 32 inputs to be our loss
loss = -prob[torch.arange(32), Y].log().mean()
print(loss.item()) # This is to be minimized

* The following is a simplified version of the code provided above:








In [ ]:
X.shape, Y.shape # dataset

In [ ]:
# Let's re-initialize the weights and biases
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27,2), generator=g)        # 27 characters, has2 dimensions each
W1 = torch.randn((6,100), generator=g)      # 3 characters, each with 2 embedding values, are used as inputs to 100 neurons.
b1 = torch.randn((100), generator=g)        # 100 biases added to the 100 neuron outputs
W2 = torch.randn((100,27), generator=g)     # 100 neuron outputs as inputs to 27 output neurons
b2 = torch.randn((27), generator=g)         # 27 biases added to the 27 output neurons

parameters = [C, W1, b1, W2, b2]            # Group all parameters into a single structure.

In [ ]:
sum(p.nelement() for p in parameters) # number of parameters in total

In [ ]:
emb = C[X] # (32,3,2)
h = torch.tanh(emb.view(-1,6) @ W1 +b1) # (32,100)
logits = h @ W2 + b2 # (32,27)
# counts = logits.exp()
# prob = counts / counts.sum(1, keepdim=True)
# loss = -prob[torch.arange(32), Y].log().mean()
loss=F.cross_entropy(logits, Y)
loss

* this is just classification. Many people use classification, which is why PyTorch provides the functional.cross_entropy function to compute this more efficiently. We can simply call f.cross_entropy, passing in the logits and the target array y, and it will calculate the same loss. In fact, we can replace the custom implementation with this function and achieve the same result.
* There are several good reasons to prefer f.cross_entropy over a custom implementation. Firstly, using f.cross_entropy prevents PyTorch from creating numerous intermediate tensors, which can be inefficient. Instead, PyTorch optimizes operations, often using fused kernels that evaluate expressions more efficiently. Secondly, the backward pass is more efficient with f.cross_entropy, not just because of fused kernels but also due to its simpler analytical and mathematical implementation.
 * Secondly, f.cross_entropy is often more numerically stable. For example, consider logits of -2, 3, -3, 0, and 5. When we take the exponent of these logits and normalize them to sum to 1, it shows how f.cross_entropy handles numerical stability effectively.If the value of our input numbers changes significantly, the result will be undesirable.
An example is given in the cell below
* There are many good reasons to use cross_entropy. Firstly, the forward pass can be much more efficient. Secondly, the backward pass can also be more efficient. Additionally, numerical stability is significantly improved.

In [ ]:
logits = torch.tensor([-2,3,-3,0,5])
counts = logits.exp()
probs = counts / counts.sum()
print(f'first probs is {probs}')
logits = torch.tensor([-2,3,-3,0,100])
counts = logits.exp()
probs = counts / counts.sum()
print(f'second probs is {probs}')

In [ ]:
for p in parameters:
    p.requires_grad = True # important !!!

In [ ]:
# lets build again..
for _ in range (10):
    #forward pass
    emb = C[X] # (32,3,2)
    h = torch.tanh(emb.view(-1,6) @ W1 +b1) # (32,100)
    logits = h @ W2 + b2 # (32,27)
    loss=F.cross_entropy(logits, Y)
    print(loss.item())
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    for p in parameters:
        p.data += -0.1 * p.grad

* If our iterator value were 1000, we would encounter an overfitting situation because we have many parameters (3,481 in this case) but only 32 examples.

In [ ]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27,2), generator=g)        # 27 characters, has2 dimensions each
W1 = torch.randn((6,100), generator=g)      # 3 characters, each with 2 embedding values, are used as inputs to 100 neurons.
b1 = torch.randn((100), generator=g)        # 100 biases added to the 100 neuron outputs
W2 = torch.randn((100,27), generator=g)     # 100 neuron outputs as inputs to 27 output neurons
b2 = torch.randn((27), generator=g)         # 27 biases added to the 27 output neurons

parameters = [C, W1, b1, W2, b2] 
for p in parameters:
    p.requires_grad = True # important !!!
for _ in range (1000):
    #forward pass
    emb = C[X] # (32,3,2)
    h = torch.tanh(emb.view(-1,6) @ W1 +b1) # (32,100)
    logits = h @ W2 + b2 # (32,27)
    loss=F.cross_entropy(logits, Y)
    
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    for p in parameters:
        p.data += -0.1 * p.grad
print(loss.item())

We're unable to achieve exactly zero loss. To understand why, let's examine the logits being predicted. By looking at the maximum values along the first dimension in PyTorch, we can see both the maximum values and their indices. The indices are often close to the labels but sometimes differ. For example, the predicted index might be 19 while the label is 5. This discrepancy occurs because the first example may be supposed to predict 'e,' but it could also predict 'o,' 'i,' or 's.' Thus, multiple outcomes are possible for the same input, preventing us from completely overfitting and achieving zero loss. However, when there is a unique input for a unique output, we overfit and get the correct result. Now, we just need to ensure we read in the full dataset and optimize the neural network.

In [ ]:
logits.max(1)

In [ ]:


# build the dataset and run with ALL dataset
block_size = 3 # context length: how many characters do we take to predict the next one?
# to guess the letter after the 3rd letter

X, Y = [], []
for w in words:

    
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        # print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

print(X.shape,Y.shape)


g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27,2), generator=g)        # 27 characters, has2 dimensions each
W1 = torch.randn((6,100), generator=g)      # 3 characters, each with 2 embedding values, are used as inputs to 100 neurons.
b1 = torch.randn((100), generator=g)        # 100 biases added to the 100 neuron outputs
W2 = torch.randn((100,27), generator=g)     # 100 neuron outputs as inputs to 27 output neurons
b2 = torch.randn((27), generator=g)         # 27 biases added to the 27 output neurons
parameters = [C, W1, b1, W2, b2] 

print(sum(p.nelement() for p in parameters))  # number of parameters

for p in parameters:
    p.requires_grad = True


In [ ]:
for _ in range (10):
    #forward pass
    
    emb = C[X] # 
    h = torch.tanh(emb.view(-1,6) @ W1 +b1) # (32,100)
    logits = h @ W2 + b2 # (32,27)
    loss=F.cross_entropy(logits, Y)
    print(loss.item())
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    for p in parameters:
        p.data += -0.1 * p.grad


* You'll notice that each iteration takes quite a bit of time because we're processing 220,000 examples in each forward and backward pass. To address this, we can use mini-batching. In practice, people perform forward and backward passes and updates on smaller batches of data. We'll randomly select a portion of the dataset as a mini-batch, then perform the forward, backward, and update steps on just that mini-batch, iterating over these mini-batches.
* when we apply arrangements we have to always define parameters. Because when we trained model all parameters are tuned. so this is not good for fair comparison or see the difference in result


In [ ]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27,2), generator=g)        # 27 characters, has2 dimensions each
W1 = torch.randn((6,100), generator=g)      # 3 characters, each with 2 embedding values, are used as inputs to 100 neurons.
b1 = torch.randn((100), generator=g)        # 100 biases added to the 100 neuron outputs
W2 = torch.randn((100,27), generator=g)     # 100 neuron outputs as inputs to 27 output neurons
b2 = torch.randn((27), generator=g)         # 27 biases added to the 27 output neurons
parameters = [C, W1, b1, W2, b2] 

print(sum(p.nelement() for p in parameters))  # number of parameters

for p in parameters:
    p.requires_grad = True

In [ ]:
for _ in range (10):
    #forward pass
    
    # minibatch construct
    ix = torch.randint(0,X.shape[0], (32,))
    emb = C[X[ix]] # (32,3,2)
    h = torch.tanh(emb.view(-1,6) @ W1 +b1) # (32,100)
    logits = h @ W2 + b2 # (32,27)
    loss=F.cross_entropy(logits, Y[ix])
    print(loss.item())
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    for p in parameters:
        p.data += -0.1 * p.grad

* This way, we can run many examples quickly and decrease the loss much faster. Although the quality of our gradient is lower when dealing with mini-batches, the direction is still useful even when estimated from only 32 examples. It's better to have an approximate gradient and take more steps than to evaluate the exact gradient and take fewer steps. This approach works well in practice
* lets look back learning rate  optimziation, first of all we reset the all parameters

In [ ]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27,2), generator=g)        # 27 characters, has2 dimensions each
W1 = torch.randn((6,100), generator=g)      # 3 characters, each with 2 embedding values, are used as inputs to 100 neurons.
b1 = torch.randn((100), generator=g)        # 100 biases added to the 100 neuron outputs
W2 = torch.randn((100,27), generator=g)     # 100 neuron outputs as inputs to 27 output neurons
b2 = torch.randn((27), generator=g)         # 27 biases added to the 27 output neurons
parameters = [C, W1, b1, W2, b2] 

print(sum(p.nelement() for p in parameters))  # number of parameters

for p in parameters:
    p.requires_grad = True

In [ ]:
lre = torch.linspace(-3,0, 1000)
lrs = 10 ** lre
lrs

In [ ]:
lr_i = []
loss_i = []
for i in range (1000):
    #forward pass
    
    # minibatch construct
    ix = torch.randint(0,X.shape[0], (32,))
    emb = C[X[ix]] # (32,3,2)
    h = torch.tanh(emb.view(-1,6) @ W1 +b1) # (32,100)
    logits = h @ W2 + b2 # (32,27)
    loss=F.cross_entropy(logits, Y[ix])
    print(loss.item())
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    lr = lrs[i]
    for p in parameters:
        p.data += -lr * p.grad
    # track stats
    lr_i.append(lre[i]) 
    loss_i.append(loss.item())

*  The reason why we use Lre[i] instead of lr  is that the values ​​to the power of 10 are written on the x-axis in the graph we will draw.
 * lr_i.append(lre[i])

In [ ]:
plt.plot(lr_i,loss_i)

* A learning rate around  0.1 is usually a good setting. In our initial setup, 0.1 was a fairly effective learning rate. This is roughly how you would determine it.

In [ ]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27,2), generator=g)        # 27 characters, has2 dimensions each
W1 = torch.randn((6,100), generator=g)      # 3 characters, each with 2 embedding values, are used as inputs to 100 neurons.
b1 = torch.randn((100), generator=g)        # 100 biases added to the 100 neuron outputs
W2 = torch.randn((100,27), generator=g)     # 100 neuron outputs as inputs to 27 output neurons
b2 = torch.randn((27), generator=g)         # 27 biases added to the 27 output neurons
parameters = [C, W1, b1, W2, b2] 

print(sum(p.nelement() for p in parameters))  # number of parameters

for p in parameters:
    p.requires_grad = True

In [ ]:

for i in range (1000):
    #forward pass
    
    # minibatch construct
    ix = torch.randint(0,X.shape[0], (32,))
    emb = C[X[ix]] # (32,3,2)
    h = torch.tanh(emb.view(-1,6) @ W1 +b1) # (32,100)
    logits = h @ W2 + b2 # (32,27)
    loss=F.cross_entropy(logits, Y[ix])
    print(loss.item())
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    lr = 0.1
    for p in parameters:
        p.data += -lr * p.grad
    # track stats
 

* You first find a decent learning rate using the approach in the code above. Then, you start with that learning rate and train for a while. Toward the end, you can apply learning rate decay, reducing the learning rate by a factor of 10 and taking a few more steps. This helps in getting a well-trained network.


* As you add more neurons and parameters, the neural network's capacity to overfit the training set increases. This means the loss on the training set can become very low, even zero, indicating that the model is merely memorizing the training data. Consequently, when you sample from the model, it will reproduce the training data exactly, without generating any new data. Moreover, the loss on a separate evaluation set may be very high, revealing that the model does not generalize well. To address this, it is standard practice to split your dataset into three parts: the training set, the development (validation) set, and the test set.
* Eighty percent of your dataset is used for the training set to optimize the model's parameters through gradient descent. Ten percent is used for the development (or validation) split to tune the hyperparameters, such as the hidden layer size, embedding size, and regularization strength. You can try different settings and see which works best on the validation split. The training split trains the parameters, while the validation split helps tune the hyperparameters. The test split, the remaining ten percent, evaluates the model's performance at the end. You should only evaluate the loss on the test set sparingly, as frequent evaluation can lead to overfitting. Let's split our training data into train, dev, and test sets, then train on the train set and evaluate on the test set very sparingly.

In [ ]:
# training split, valid siplit and test split
# %80, %10, %10
# build the dataset

def build_dataset(words):
  block_size = 3 # context length: how many characters do we take to predict the next one?

  X, Y = [], []
  for w in words:

    #print(w)
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      #print(''.join(itos[i] for i in context), '--->', itos[ix])
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtrain, Ytrain = build_dataset(words[:n1])
Xvalid, Yvalid = build_dataset(words[n1:n2])
Xtest, Ytest = build_dataset(words[n2:])

In [ ]:
print(f'train samples: {Xtrain.shape[0]}')
print(f'train samples: {Xvalid.shape[0]}')
print(f'train samples: {Xtest.shape[0]}')

In [ ]:
Xtrain.shape, Ytrain.shape

In [ ]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27,2), generator=g)        # 27 characters, has2 dimensions each
W1 = torch.randn((6,100), generator=g)      # 3 characters, each with 2 embedding values, are used as inputs to 100 neurons.
b1 = torch.randn((100), generator=g)        # 100 biases added to the 100 neuron outputs
W2 = torch.randn((100,27), generator=g)     # 100 neuron outputs as inputs to 27 output neurons
b2 = torch.randn((27), generator=g)         # 27 biases added to the 27 output neurons
parameters = [C, W1, b1, W2, b2] 

print(sum(p.nelement() for p in parameters))  # number of parameters

for p in parameters:
    p.requires_grad = True

In [ ]:
lre = torch.linspace(-3,0, 1000)
lrs = 10 ** lre


In [ ]:
lr_i = []
loss_i = []
for i in range (30000):
    #forward pass
    
    # minibatch construct
    ix = torch.randint(0,Xtrain.shape[0], (32,))
    emb = C[Xtrain[ix]] # (32,3,2)
    h = torch.tanh(emb.view(-1,6) @ W1 +b1) # (32,100)
    logits = h @ W2 + b2 # (32,27)
    loss=F.cross_entropy(logits, Ytrain[ix])
    
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    # lr = lrs[i]
    lr = 0.1
    for p in parameters:
        p.data += -lr * p.grad

    # # track stats
    # lr_i.append(lre[i]) 
    # loss_i.append(loss.item())

In [ ]:


print(loss.item())

In [ ]:
# loss on train set
emb = C[Xtrain]
h = torch.tanh(emb.view(-1,6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytrain)
loss

In [ ]:
# loss on valid set
emb = C[Xvalid]
h = torch.tanh(emb.view(-1,6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Yvalid)
loss

* We see that the training and dev losses are about equal, indicating that we're not overfitting. This model isn't powerful enough to purely memorize the data, and we are actually underfitting because the training and dev/test losses are roughly equal. This usually means our network is too small. To improve performance, we should scale up the size of the neural net. Let's increase the size of the neural net to address this.
* The easiest way to scale up is to increase the number of neurons in the hidden layer. Currently, it has 100 neurons, so let's increase it to 300. This means we'll also have 300 biases, and 300 inputs into the final layer. After initializing our neural net with these changes, we'll have ten thousand parameters instead of three thousand.

In [ ]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27,2), generator=g)        # 27 characters, has2 dimensions each
W1 = torch.randn((6,300), generator=g)      # 3 characters, each with 2 embedding values, are used as inputs to 100 neurons.
b1 = torch.randn((300), generator=g)        # 100 biases added to the 100 neuron outputs
W2 = torch.randn((300,27), generator=g)     # 100 neuron outputs as inputs to 27 output neurons
b2 = torch.randn((27), generator=g)         # 27 biases added to the 27 output neurons
parameters = [C, W1, b1, W2, b2] 

print(sum(p.nelement() for p in parameters))  # number of parameters

for p in parameters:
    p.requires_grad = True

In [ ]:
lr_i = []
loss_i = []
stepi = []
for i in range (70000):
    #forward pass
    
    # minibatch construct
    ix = torch.randint(0,Xtrain.shape[0], (32,))
    emb = C[Xtrain[ix]] # (32,3,2)
    h = torch.tanh(emb.view(-1,6) @ W1 +b1) # (32,100)
    logits = h @ W2 + b2 # (32,27)
    loss=F.cross_entropy(logits, Ytrain[ix])
    
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    # lr = lrs[i]
    lr = 0.05
    for p in parameters:
        p.data += -lr * p.grad

    # # track stats
    stepi.append(i) 
    loss_i.append(loss.item())

In [ ]:
plt.plot(stepi,loss_i)

In [ ]:
# loss on train set
emb = C[Xtrain]
h = torch.tanh(emb.view(-1,6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytrain)
loss

In [ ]:
# loss on valid set
emb = C[Xvalid]
h = torch.tanh(emb.view(-1,6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Yvalid)
loss

In [ ]:
# loss on train set
emb = C[Xtrain]
h = torch.tanh(emb.view(-1,6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytrain)
loss

In [ ]:
# loss on valid set
emb = C[Xvalid]
h = torch.tanh(emb.view(-1,6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Yvalid)
loss

* These results are better than the results of the small size model. The reason why we increased the number of epochs is that the model needs more epochs as the number of parameters increases.
 * We can try different things to reduce the loss. one of them is embedding size increase

In [ ]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27,10), generator=g)        # 27 characters, has 10 dimensions each
W1 = torch.randn((30,200), generator=g)      # 3 characters, each with 10 embedding values, are used as inputs to 200 neurons.
b1 = torch.randn((200), generator=g)        # 200 biases added to the 200 neuron outputs
W2 = torch.randn((200,27), generator=g)     # 200 neuron outputs as inputs to 27 output neurons
b2 = torch.randn((27), generator=g)         # 27 biases added to the 27 output neurons
parameters = [C, W1, b1, W2, b2] 

print(sum(p.nelement() for p in parameters))  # number of parameters

for p in parameters:
    p.requires_grad = True

In [ ]:
lr_i = []
loss_i = []
stepi = []
for i in range (100000):
    #forward pass
    
    # minibatch construct
    ix = torch.randint(0,Xtrain.shape[0], (32,))
    emb = C[Xtrain[ix]] # (32,3,2)
    h = torch.tanh(emb.view(-1,30) @ W1 +b1) # its 30 because 3 characters, each with 10 embedding values 3*10
    logits = h @ W2 + b2 # (32,27)
    loss=F.cross_entropy(logits, Ytrain[ix])
    
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    # lr = lrs[i]
    lr = 0.01
    for p in parameters:
        p.data += -lr * p.grad

    # # track stats
    stepi.append(i) 
    loss_i.append(loss.log10().item()) # it will looking good on plot

In [ ]:
plt.plot(stepi,loss_i)

* its looking thick because of bath size we can decrease it

In [ ]:
# loss on train set
emb = C[Xtrain]
h = torch.tanh(emb.view(-1,30) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytrain)
loss

In [ ]:
# loss on valid set
emb = C[Xvalid]
h = torch.tanh(emb.view(-1,30) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Yvalid)
loss

* the clean code in above

In [ ]:
for i in range(200000):
  
  # minibatch construct
  ix = torch.randint(0, Xtrain.shape[0], (32,))
  
  # forward pass
  emb = C[Xtrain[ix]] # (32, 3, 2)
  h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
  logits = h @ W2 + b2 # (32, 27)
  loss = F.cross_entropy(logits, Ytrain[ix])
  #print(loss.item())
  
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  #lr = lrs[i]
  lr = 0.1 if i < 100000 else 0.01
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  #lri.append(lre[i])
  stepi.append(i)
  loss_i.append(loss.log10().item())

#print(loss.item())

In [ ]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))